In [2]:
import cv2
import mediapipe as mp
import numpy as np

In [3]:
BaseOptions = mp.tasks.BaseOptions
PoseLandmarker = mp.tasks.vision.PoseLandmarker
PoseLandmarkerOptions = mp.tasks.vision.PoseLandmarkerOptions
RunningMode = mp.tasks.vision.RunningMode

In [4]:
model_path = "pose_landmarker_full.task"

In [5]:
options = PoseLandmarkerOptions(
    base_options=BaseOptions(
        model_asset_path="pose_landmarker_full.task"
    ),
    running_mode=RunningMode.VIDEO,
    num_poses=1,
    min_pose_detection_confidence=0.5,
    min_pose_presence_confidence=0.5,
    min_tracking_confidence=0.5
)

In [13]:
import math

def calculate_angle(a, b, c):

    angle = math.degrees(
        math.atan2(c[1] - b[1], c[0] - b[0])
        - math.atan2(a[1] - b[1], a[0] - b[0])
    )

    angle = abs(angle)

    if angle > 180:
        angle = 360 - angle

    return angle

In [15]:
UP_THRESHOLD = 50
DOWN_THRESHOLD = 160

In [22]:
with PoseLandmarker.create_from_options(options) as landmarker:

    cap = cv2.VideoCapture(0)

    timestamp_ms = 0
    stage = None
    count = 0
    while cap.isOpened():

        ret, frame = cap.read()

        if not ret:
            break

        rgb_frame = cv2.cvtColor(
            frame,
            cv2.COLOR_BGR2RGB
        )

        mp_image = mp.Image(
            image_format=mp.ImageFormat.SRGB,
            data=rgb_frame
        )

        results = landmarker.detect_for_video(
            mp_image,
            timestamp_ms
        )

        timestamp_ms += 33

        if results.pose_landmarks:

            landmarks = results.pose_landmarks[0]

            shoulder = landmarks[12]
            elbow = landmarks[14]
            wrist = landmarks[16]

            h, w = frame.shape[:2]

            shoulder_point = (
                int(shoulder.x * w),
                int(shoulder.y * h)
            )

            elbow_point = (
            int(elbow.x * w),
            int(elbow.y * h)
            )

            wrist_point = (
            int(wrist.x * w),
            int(wrist.y * h)
            )

            angle = calculate_angle(
                shoulder_point,
                elbow_point,
                wrist_point)
            if angle < UP_THRESHOLD:
                stage = "UP"

            if angle > DOWN_THRESHOLD and stage == "UP":
                stage = "DOWN"
                count += 1
            cv2.putText(
                frame,
                str(int(angle)),
                elbow_point,
                cv2.FONT_HERSHEY_SIMPLEX,
                1,
                (0, 0, 255),
                2
            )
            cv2.putText(
                frame,
                stage,
                (50, 50),
                cv2.FONT_HERSHEY_SIMPLEX,
                1,
                (0, 0, 255),
                2
            )
            cv2.putText(
                frame,
                str(count),
                (50, 100),
                cv2.FONT_HERSHEY_SIMPLEX,
                1,
                (0, 0, 255),
                2
            )
            for landmark in landmarks:

                x = int(landmark.x * frame.shape[1])
                y = int(landmark.y * frame.shape[0])

                cv2.circle(
                    frame,
                    (x, y),
                    5,
                    (0, 255, 0),
                    -1
                )

            connections = (
                mp.tasks.vision.PoseLandmarksConnections.POSE_LANDMARKS
            )

            for connection in connections:

                start = landmarks[connection.start]
                end = landmarks[connection.end]

                x1 = int(start.x * frame.shape[1])
                y1 = int(start.y * frame.shape[0])

                x2 = int(end.x * frame.shape[1])
                y2 = int(end.y * frame.shape[0])

                cv2.line(
                    frame,
                    (x1, y1),
                    (x2, y2),
                    (0, 255, 0),
                    2
                )

        cv2.imshow("Pose Estimation", frame)

        if cv2.waitKey(5) & 0xFF == 27:
            break

    cap.release()
    cv2.destroyAllWindows()

W0000 00:00:1788808424.930531   24659 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1788808424.943320   24660 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
